# Chapter 9: Reinforcement Learning for LLMs

Covers RLHF, PPO, DPO, and GRPO from mathematical first principles.

**Topics covered:**
1. MDP Formulation for LLMs
2. Policy Gradient & REINFORCE
3. PPO Clipped Objective
4. Generalized Advantage Estimation (GAE)
5. Reward Modeling (Bradley-Terry)
6. KL Constraint in RLHF
7. Direct Preference Optimization (DPO)
8. Group Relative Policy Optimization (GRPO)
9. Comparison: RLHF vs DPO vs GRPO

**Install:** `!pip install torch` if needed.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

torch.manual_seed(42)
print(f"PyTorch {torch.__version__}")

## 1. MDP Formulation for LLMs

Language model generation maps naturally onto a **Markov Decision Process (MDP)**:

| MDP Component | LLM Interpretation |
|---|---|
| **Policy** $\pi_\theta$ | The LLM parameterized by $\theta$ |
| **State** $s_t$ | Token sequence generated so far: $(x, y_1, \ldots, y_{t-1})$ |
| **Action** $a_t$ | Next token sampled: $a_t \sim \pi_\theta(\cdot \mid s_t)$ |
| **Reward** $R_t$ | Sparse: 0 for intermediate tokens, $r(x, y)$ at end-of-sequence |
| **Transition** | Deterministic: $s_{t+1} = (s_t, a_t)$ |

### Discounted Return

The **return** (cumulative discounted reward) from time $t$ is:

$$G_t = \sum_{k=0}^{\infty} \gamma^k R_{t+k}$$

where $\gamma \in [0, 1]$ is the discount factor. For LLMs with sparse end-of-sequence reward, $\gamma = 1$ is common.

### Value Function

$$V^\pi(s) = \mathbb{E}_\pi[G_t \mid s_t = s]$$

The value function estimates the expected return from state $s$ under policy $\pi$. It is used as a **baseline** to reduce variance in policy gradient methods.

In [ ]:
class PolicyNetwork(nn.Module):
    """Small transformer-like policy network.
    Input: token sequence embeddings (B, T, d_model)
    Output: logits over vocabulary (B, T, vocab_size)
    """
    def __init__(self, vocab_size=50, d_model=32, nhead=4, num_layers=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=64,
            batch_first=True, norm_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, token_ids):
        # token_ids: (B, T)
        T = token_ids.shape[1]
        # Causal mask: upper triangular
        causal_mask = torch.triu(torch.ones(T, T, device=token_ids.device), diagonal=1).bool()
        x = self.embedding(token_ids)                          # (B, T, d_model)
        x = self.transformer(x, mask=causal_mask, is_causal=True)  # (B, T, d_model)
        logits = self.lm_head(x)                               # (B, T, vocab_size)
        return logits


class ValueNetwork(nn.Module):
    """Value head estimating V(s_t) from hidden states."""
    def __init__(self, d_model=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, hidden_states):
        # hidden_states: (B, T, d_model)
        return self.net(hidden_states).squeeze(-1)   # (B, T)


# Instantiate
vocab_size = 50
d_model = 32
policy = PolicyNetwork(vocab_size=vocab_size, d_model=d_model)
value_net = ValueNetwork(d_model=d_model)

print(f"Policy parameters:     {sum(p.numel() for p in policy.parameters()):,}")
print(f"Value net parameters:  {sum(p.numel() for p in value_net.parameters()):,}")

# Sample an action (next token) from policy logits
B, T = 2, 5
input_ids = torch.randint(0, vocab_size, (B, T))
logits = policy(input_ids)                            # (B, T, vocab_size)
probs = F.softmax(logits[:, -1, :], dim=-1)           # last-position probs: (B, vocab_size)
next_token = torch.multinomial(probs, num_samples=1)  # (B, 1)
log_prob = torch.log(probs.gather(1, next_token))      # (B, 1)

print(f"\nInput shape:        {input_ids.shape}")
print(f"Logits shape:       {logits.shape}")
print(f"Next token sampled: {next_token.squeeze().tolist()}")
print(f"Log prob of sample: {log_prob.squeeze().tolist()}")

## 2. Policy Gradient & REINFORCE

### Policy Gradient Theorem

The objective is to maximize expected return $J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}[G_0]$. The gradient is:

$$\nabla_\theta J(\theta) = \mathbb{E}\left[\sum_t \nabla_\theta \log \pi_\theta(a_t \mid s_t) \cdot G_t\right]$$

This is the **log-derivative trick**: $\nabla_\theta \pi_\theta = \pi_\theta \nabla_\theta \log \pi_\theta$.

### Baseline for Variance Reduction

Subtracting a baseline $b(s_t)$ does not bias the gradient (since $\mathbb{E}[\nabla_\theta \log \pi_\theta \cdot b] = 0$), but reduces variance:

$$\nabla_\theta J(\theta) = \mathbb{E}\left[\sum_t \nabla_\theta \log \pi_\theta(a_t \mid s_t) \cdot (G_t - V(s_t))\right]$$

The **advantage** $A_t = G_t - V(s_t)$ measures how much better action $a_t$ is compared to the average. The REINFORCE loss (to minimize) is:

$$\mathcal{L}_{REINFORCE} = -\sum_t \log \pi_\theta(a_t \mid s_t) \cdot A_t$$

Minimizing this loss via gradient descent ascends $J(\theta)$.

In [ ]:
def compute_returns(rewards, gamma=0.99):
    """Compute discounted returns G_t for a sequence of rewards.
    Args:
        rewards: list or tensor of shape (T,)
        gamma: discount factor
    Returns:
        returns: tensor of shape (T,)
    """
    T = len(rewards)
    returns = torch.zeros(T)
    G = 0.0
    for t in reversed(range(T)):
        G = rewards[t] + gamma * G
        returns[t] = G
    return returns


def reinforce_loss(log_probs, rewards, baseline_values=None, gamma=0.99):
    """REINFORCE policy gradient loss.
    Args:
        log_probs: (T,) log probabilities of actions taken
        rewards: (T,) rewards received
        baseline_values: (T,) value estimates V(s_t), or None
        gamma: discount factor
    Returns:
        loss: scalar (negative expected return)
    """
    returns = compute_returns(rewards, gamma)     # G_t for each t

    if baseline_values is not None:
        advantages = returns - baseline_values.detach()
    else:
        advantages = returns

    # Normalize advantages for stability (optional but common)
    advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

    # Policy gradient loss: -sum( log_pi * A_t )
    policy_loss = -(log_probs * advantages).sum()
    return policy_loss, returns, advantages


# Demo on toy 5-token sequence
T = 5
torch.manual_seed(42)

# Simulated log probabilities of selected actions (tokens)
log_probs = torch.tensor([-1.2, -0.8, -2.1, -0.5, -1.5], requires_grad=True)

# Sparse reward: only last token gets reward (e.g., +1 for good response)
rewards = [0.0, 0.0, 0.0, 0.0, 1.0]

# Simulated value estimates
baseline_values = torch.tensor([0.3, 0.4, 0.5, 0.6, 0.7])

loss_no_baseline, returns_no_base, adv_no_base = reinforce_loss(log_probs, rewards, baseline_values=None)
loss_with_baseline, returns_with_base, adv_with_base = reinforce_loss(log_probs, rewards, baseline_values=baseline_values)

print("Toy 5-token sequence REINFORCE:")
print(f"  Rewards:          {rewards}")
print(f"  Returns G_t:      {returns_no_base.tolist()}")
print(f"  Advantages (no baseline):   {adv_no_base.round(decimals=3).tolist()}")
print(f"  Advantages (with baseline): {adv_with_base.round(decimals=3).tolist()}")
print(f"\n  Loss (no baseline):   {loss_no_baseline.item():.4f}")
print(f"  Loss (with baseline): {loss_with_baseline.item():.4f}")

# Verify gradient flows through log_probs
loss_with_baseline.backward()
print(f"\n  Gradient of log_probs: {log_probs.grad.round(decimals=3).tolist()}")
print("  (negative grad => increasing these log-probs decreases loss => policy update)")

## 3. PPO Clipped Objective

**Proximal Policy Optimization (PPO)** addresses the instability of vanilla policy gradient by constraining how much the policy can change in a single update.

### Probability Ratio

$$r_t(\theta) = \frac{\pi_\theta(a_t \mid s_t)}{\pi_{\text{old}}(a_t \mid s_t)} = \exp(\log \pi_\theta - \log \pi_{\text{old}})$$

When $\theta = \theta_{\text{old}}$: $r_t = 1$. When the policy moves away, $r_t \neq 1$.

### Clipped Surrogate Objective

$$\mathcal{L}^{\text{CLIP}}(\theta) = \mathbb{E}_t\left[\min\left(r_t(\theta)\hat{A}_t,\ \text{clip}(r_t(\theta), 1-\varepsilon, 1+\varepsilon)\hat{A}_t\right)\right]$$

The clip prevents the policy from taking excessively large steps:
- If $\hat{A}_t > 0$ (good action): cap $r_t$ at $1 + \varepsilon$ — don't over-exploit
- If $\hat{A}_t < 0$ (bad action): floor $r_t$ at $1 - \varepsilon$ — don't over-punish

Typical $\varepsilon = 0.1$–$0.2$. Combined with multiple epochs of gradient updates, this is the workhorse algorithm for RLHF training.

### Full PPO Loss

$$\mathcal{L}^{PPO} = \mathcal{L}^{\text{CLIP}} - c_1 \mathcal{L}^{VF} + c_2 \mathcal{H}[\pi_\theta]$$

where $\mathcal{L}^{VF}$ is the value function loss (MSE) and $\mathcal{H}$ is an entropy bonus.

In [ ]:
def ppo_clip_loss(old_log_probs, new_log_probs, advantages, epsilon=0.2):
    """Compute PPO clipped surrogate loss.
    Args:
        old_log_probs: (T,) or (B, T) log probs under pi_old
        new_log_probs: (T,) or (B, T) log probs under pi_theta (current)
        advantages: (T,) or (B, T) advantage estimates
        epsilon: clipping threshold (typically 0.1-0.2)
    Returns:
        loss: scalar (to minimize; negated clip objective)
        ratio: probability ratios
        clipped_frac: fraction of ratios that were clipped
    """
    # Probability ratio: r_t = pi_theta / pi_old
    ratio = torch.exp(new_log_probs - old_log_probs)

    # Unclipped objective: r_t * A_t
    obj_unclipped = ratio * advantages

    # Clipped objective: clip(r_t, 1-eps, 1+eps) * A_t
    ratio_clipped = torch.clamp(ratio, 1.0 - epsilon, 1.0 + epsilon)
    obj_clipped = ratio_clipped * advantages

    # PPO objective: min of unclipped and clipped
    clip_obj = torch.min(obj_unclipped, obj_clipped)

    # Fraction of ratios that hit the clip bounds
    clipped_frac = ((ratio < 1.0 - epsilon) | (ratio > 1.0 + epsilon)).float().mean()

    # Negate for minimization
    loss = -clip_obj.mean()
    return loss, ratio, clipped_frac


# Demo: show how clipping limits policy change
torch.manual_seed(42)
T = 10
old_log_probs = torch.randn(T) - 1.0        # simulate old policy
advantages = torch.randn(T)                  # some positive, some negative

print("PPO Clipping Demo — varying policy update size:")
print(f"{'Update scale':>14} | {'Loss':>8} | {'Clip frac':>10} | {'Ratio range':<20}")
print("-" * 60)

for scale in [0.0, 0.05, 0.1, 0.2, 0.5, 1.0, 2.0]:
    # Simulate policy moving by 'scale' in log-prob space
    new_log_probs = old_log_probs + torch.randn(T) * scale
    loss, ratio, clip_frac = ppo_clip_loss(old_log_probs, new_log_probs, advantages, epsilon=0.2)
    r_min, r_max = ratio.min().item(), ratio.max().item()
    print(f"{scale:>14.2f} | {loss.item():>8.4f} | {clip_frac.item():>10.3f} | [{r_min:.3f}, {r_max:.3f}]")

print("\nAs policy update scale grows, more ratios are clipped -> gradient signal dampened.")

## 4. Generalized Advantage Estimation (GAE)

GAE (Schulman et al., 2016) provides a principled interpolation between high-variance Monte Carlo returns and high-bias TD(0) estimates.

### TD Residual

The one-step TD error ("delta"):

$$\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$$

This is the "surprise" — how much better the actual outcome was compared to the value prediction.

### GAE Formula

$$\hat{A}_t^{GAE(\gamma, \lambda)} = \sum_{l=0}^{\infty} (\gamma \lambda)^l \delta_{t+l}$$

Implemented recursively: $\hat{A}_t = \delta_t + \gamma \lambda \hat{A}_{t+1}$

### Bias-Variance Tradeoff

| $\lambda$ | Equivalent to | Bias | Variance |
|---|---|---|---|
| $\lambda = 0$ | TD(0): $\hat{A}_t = \delta_t$ | High bias | Low variance |
| $\lambda = 1$ | Monte Carlo: $\hat{A}_t = \sum_l \gamma^l r_{t+l} - V(s_t)$ | Low bias | High variance |
| $0 < \lambda < 1$ | Exponentially weighted | Balanced | Balanced |

PPO typically uses $\gamma = 0.99$, $\lambda = 0.95$.

In [ ]:
def compute_gae(rewards, values, gamma=0.99, lam=0.95, last_value=0.0):
    """Compute Generalized Advantage Estimation.
    Args:
        rewards: (T,) rewards for each step
        values:  (T,) value estimates V(s_t) for each step
        gamma:   discount factor
        lam:     GAE lambda parameter
        last_value: V(s_{T+1}), value of terminal/next state (0 for episode end)
    Returns:
        advantages: (T,) GAE advantage estimates
        returns:    (T,) value targets = advantages + values
    """
    T = len(rewards)
    advantages = torch.zeros(T)
    gae = 0.0

    # Extend values with last_value for bootstrapping
    values_ext = torch.cat([values, torch.tensor([last_value])])

    for t in reversed(range(T)):
        delta = rewards[t] + gamma * values_ext[t + 1] - values_ext[t]
        gae = delta + gamma * lam * gae
        advantages[t] = gae

    returns = advantages + values   # value targets for critic update
    return advantages, returns


# Demo on random reward sequence
torch.manual_seed(42)
T = 8
rewards = torch.randn(T) * 0.5
rewards[-1] = 1.0   # terminal reward
values = torch.rand(T) * 0.5   # simulated value estimates

print("GAE Demo (T=8):")
print(f"  Rewards: {rewards.round(decimals=3).tolist()}")
print(f"  Values:  {values.round(decimals=3).tolist()}")
print()

for lam in [0.0, 0.5, 0.95, 1.0]:
    adv, ret = compute_gae(rewards, values, gamma=0.99, lam=lam)
    label = "TD(0)" if lam == 0.0 else ("MC" if lam == 1.0 else f"λ={lam}")
    print(f"  λ={lam:.2f} ({label:7s}): advantages = {adv.round(decimals=3).tolist()}")
    print(f"            variance of advantages: {adv.var().item():.4f}")
    print()

# TD residuals
values_ext = torch.cat([values, torch.tensor([0.0])])
deltas = rewards + 0.99 * values_ext[1:] - values_ext[:-1]
print(f"  TD residuals δ_t: {deltas.round(decimals=3).tolist()}")

## 5. Reward Modeling

In RLHF, humans provide **preference labels** rather than scalar rewards. Given a prompt $x$ and two responses $y_w$ (preferred/"winner") and $y_l$ (rejected/"loser"):

### Bradley-Terry Model

The probability that $y_w$ is preferred over $y_l$ is modeled as:

$$P(y_w \succ y_l \mid x) = \sigma(r(x, y_w) - r(x, y_l)) = \frac{e^{r(x,y_w)}}{e^{r(x,y_w)} + e^{r(x,y_l)}}$$

where $\sigma$ is the sigmoid function and $r(x, y)$ is the learned reward model.

### Reward Model Training Loss

Maximize log-likelihood of observed preferences:

$$\mathcal{L}_{RM} = -\mathbb{E}_{(x, y_w, y_l) \sim \mathcal{D}}\left[\log \sigma(r(x, y_w) - r(x, y_l))\right]$$

This is equivalent to binary cross-entropy where the "label" is always 1 (chosen is better).

### Architecture

A reward model is typically the LLM fine-tuned to output a single scalar per sequence (using a linear head on the final `[EOS]` token embedding).

In [ ]:
def reward_model_loss(r_chosen, r_rejected):
    """Bradley-Terry reward model loss.
    Args:
        r_chosen:   (B,) reward scores for preferred responses
        r_rejected: (B,) reward scores for rejected responses
    Returns:
        loss: scalar (negative log-likelihood)
        accuracy: fraction of pairs where r_chosen > r_rejected
    """
    # -log(sigma(r_w - r_l)) = -log_sigmoid(r_w - r_l)
    loss = -F.logsigmoid(r_chosen - r_rejected).mean()
    accuracy = (r_chosen > r_rejected).float().mean()
    return loss, accuracy


# Toy reward model training demo
class ToyRewardModel(nn.Module):
    """Simple reward model: embed + pool + scalar."""
    def __init__(self, vocab_size=50, d_model=16, max_len=10):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.head = nn.Linear(d_model, 1, bias=False)

    def forward(self, token_ids):
        # token_ids: (B, T)
        x = self.embedding(token_ids).mean(dim=1)  # mean pool -> (B, d_model)
        return self.head(x).squeeze(-1)              # (B,)


torch.manual_seed(42)
rm = ToyRewardModel(vocab_size=50, d_model=16)
optimizer = torch.optim.Adam(rm.parameters(), lr=1e-3)

B, T = 4, 8
print("Training toy reward model on synthetic preferences:")
print(f"{'Step':>5} | {'Loss':>8} | {'Accuracy':>10} | {'r_chosen mean':>14} | {'r_rejected mean':>16}")
print("-" * 65)

for step in range(20):
    # Synthetic: chosen responses have token IDs in [25, 50), rejected in [0, 25)
    chosen_ids   = torch.randint(25, 50, (B, T))
    rejected_ids = torch.randint(0,  25, (B, T))

    r_chosen   = rm(chosen_ids)
    r_rejected = rm(rejected_ids)

    loss, acc = reward_model_loss(r_chosen, r_rejected)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 4 == 0:
        print(f"{step:>5} | {loss.item():>8.4f} | {acc.item():>10.3f} | "
              f"{r_chosen.mean().item():>14.4f} | {r_rejected.mean().item():>16.4f}")

print("\nAs training progresses, r_chosen > r_rejected and accuracy -> 1.0")

## 6. KL Constraint in RLHF

Without constraint, RL fine-tuning can cause the policy $\pi_\theta$ to **"reward hack"** — generate text that scores high on the reward model but diverges far from natural language.

### KL-Regularized Objective

$$\max_{\pi_\theta} \mathbb{E}_{x \sim \mathcal{D}, y \sim \pi_\theta(\cdot|x)}\left[r(x, y) - \beta D_{KL}(\pi_\theta(\cdot|x) \| \pi_{\text{ref}}(\cdot|x))\right]$$

where $\pi_{\text{ref}}$ is the SFT (supervised fine-tuned) base model.

### Per-Token KL Approximation

The KL penalty at the token level is computed as:

$$\text{KL reward}(t) = -\beta \log \frac{\pi_\theta(a_t)}{\pi_{\text{ref}}(a_t)} = -\beta (\log \pi_\theta(a_t) - \log \pi_{\text{ref}}(a_t))$$

This is subtracted from the reward at each token step, effectively penalizing divergence from the reference model at each position.

### Role of $\beta$

- $\beta = 0$: pure reward maximization (can degenerate)
- $\beta \to \infty$: stays at $\pi_{\text{ref}}$ (no improvement)
- Typical range: $\beta \in [0.01, 0.5]$

In [ ]:
def kl_penalty(policy_logits, ref_logits, beta=0.1):
    """Compute per-token KL divergence penalty.
    KL(pi_theta || pi_ref) = sum_a pi_theta(a) * log(pi_theta(a) / pi_ref(a))
    For sampled tokens: approximate with log(pi_theta(a_t)) - log(pi_ref(a_t))

    Args:
        policy_logits: (B, T, V) logits from current policy
        ref_logits:    (B, T, V) logits from reference policy
        beta:          KL penalty coefficient
    Returns:
        kl_per_token: (B, T) per-token KL (exact, over full vocab)
        total_kl:     scalar mean KL
        kl_reward:    (B, T) KL penalty to subtract from reward
    """
    # Full KL over vocabulary: KL(pi || ref)
    log_pi = F.log_softmax(policy_logits, dim=-1)   # (B, T, V)
    log_ref = F.log_softmax(ref_logits, dim=-1)      # (B, T, V)
    pi = log_pi.exp()                                # (B, T, V)

    # KL(pi || ref) = sum_a pi(a) * (log_pi(a) - log_ref(a))
    kl_per_token = (pi * (log_pi - log_ref)).sum(dim=-1)  # (B, T)

    total_kl = kl_per_token.mean()
    kl_reward = -beta * kl_per_token  # negative: penalizes divergence

    return kl_per_token, total_kl, kl_reward


torch.manual_seed(42)
B, T, V = 2, 6, 50

ref_logits = torch.randn(B, T, V)  # reference (SFT) model logits

print("KL Penalty for varying policy divergence from reference:")
print(f"{'Noise scale':>12} | {'Total KL':>10} | {'Mean KL reward (β=0.1)':>24}")
print("-" * 52)

for noise_scale in [0.0, 0.1, 0.5, 1.0, 2.0, 5.0]:
    # Simulate policy that has drifted from reference
    policy_logits = ref_logits + torch.randn(B, T, V) * noise_scale
    kl_tok, total_kl, kl_rew = kl_penalty(policy_logits, ref_logits, beta=0.1)
    print(f"{noise_scale:>12.1f} | {total_kl.item():>10.4f} | {kl_rew.mean().item():>24.4f}")

print("\nAt noise_scale=0.0, policy == reference => KL=0 (no penalty)")
print("As policy drifts, KL grows => larger negative reward penalty")

# Show different beta values
policy_logits = ref_logits + torch.randn(B, T, V) * 1.0
print("\nEffect of beta on KL penalty (noise_scale=1.0):")
for beta in [0.01, 0.05, 0.1, 0.2, 0.5]:
    kl_tok, total_kl, kl_rew = kl_penalty(policy_logits, ref_logits, beta=beta)
    print(f"  β={beta:.2f}: total_kl={total_kl.item():.4f}, mean_kl_reward={kl_rew.mean().item():.4f}")

## 7. Direct Preference Optimization (DPO)

DPO (Rafailov et al., 2023) elegantly eliminates the need for a separate reward model and RL training loop.

### Key Insight

The optimal policy for the KL-regularized RLHF objective admits a closed-form solution:

$$\pi^*(y \mid x) \propto \pi_{\text{ref}}(y \mid x) \exp\!\left(\frac{r^*(x, y)}{\beta}\right)$$

Inverting this, the **implicit reward** of any policy $\pi$ relative to $\pi_{\text{ref}}$ is:

$$r^*(x, y) = \beta \log \frac{\pi^*(y \mid x)}{\pi_{\text{ref}}(y \mid x)} + \beta \log Z(x)$$

where $Z(x)$ is the partition function (constant w.r.t. $y$).

### DPO Loss

Substituting the implicit reward into the Bradley-Terry model and noting that $Z(x)$ cancels:

$$\mathcal{L}_{\text{DPO}}(\theta) = -\mathbb{E}\left[\log \sigma\!\left(\beta \log \frac{\pi_\theta(y_w \mid x)}{\pi_{\text{ref}}(y_w \mid x)} - \beta \log \frac{\pi_\theta(y_l \mid x)}{\pi_{\text{ref}}(y_l \mid x)}\right)\right]$$

### Advantages over RLHF
- No reward model needed
- No RL loop (simple supervised-style training)
- No value function
- Implicit reward: $r^* = \beta \log(\pi^*/\pi_{\text{ref}}) + \text{const}$

In [ ]:
def sequence_log_prob(logits, token_ids):
    """Compute sum of log probabilities of tokens in a sequence.
    Args:
        logits:    (B, T, V) logits from language model
        token_ids: (B, T) token IDs of the sequence
    Returns:
        log_probs: (B,) sum of log probs across sequence positions
    """
    log_probs = F.log_softmax(logits, dim=-1)          # (B, T, V)
    # Gather log prob of each selected token
    token_log_probs = log_probs.gather(2, token_ids.unsqueeze(-1)).squeeze(-1)  # (B, T)
    return token_log_probs.sum(dim=-1)  # (B,)


def dpo_loss(policy_logprobs_w, policy_logprobs_l, ref_logprobs_w, ref_logprobs_l, beta=0.1):
    """Direct Preference Optimization loss.
    Args:
        policy_logprobs_w: (B,) policy log prob of chosen response
        policy_logprobs_l: (B,) policy log prob of rejected response
        ref_logprobs_w:    (B,) reference log prob of chosen response
        ref_logprobs_l:    (B,) reference log prob of rejected response
        beta:              KL coefficient
    Returns:
        loss: scalar DPO loss
        implicit_reward_margin: (B,) r*(y_w) - r*(y_l)
    """
    # Log ratio: log(pi_theta(y) / pi_ref(y)) for chosen and rejected
    log_ratio_w = policy_logprobs_w - ref_logprobs_w
    log_ratio_l = policy_logprobs_l - ref_logprobs_l

    # Implicit reward margin: beta * (log_ratio_w - log_ratio_l)
    reward_margin = beta * (log_ratio_w - log_ratio_l)

    # DPO loss: -log(sigma(reward_margin))
    loss = -F.logsigmoid(reward_margin).mean()

    return loss, reward_margin


torch.manual_seed(42)
B, T, V = 4, 8, 50

# Simulate: policy has been trained a bit to prefer chosen responses
ref_logits_w   = torch.randn(B, T, V)
ref_logits_l   = torch.randn(B, T, V)
# Policy is ref + small update favoring chosen
policy_logits_w = ref_logits_w + 0.5 * torch.randn(B, T, V)
policy_logits_l = ref_logits_l - 0.3 * torch.randn(B, T, V)

tokens_w = torch.randint(0, V, (B, T))
tokens_l = torch.randint(0, V, (B, T))

pi_w   = sequence_log_prob(policy_logits_w, tokens_w)
pi_l   = sequence_log_prob(policy_logits_l, tokens_l)
ref_w  = sequence_log_prob(ref_logits_w,    tokens_w)
ref_l  = sequence_log_prob(ref_logits_l,    tokens_l)

loss, reward_margin = dpo_loss(pi_w, pi_l, ref_w, ref_l, beta=0.1)

print("DPO Loss Demo:")
print(f"  Policy log prob (chosen):   {pi_w.tolist()}")
print(f"  Policy log prob (rejected): {pi_l.tolist()}")
print(f"  Ref    log prob (chosen):   {ref_w.tolist()}")
print(f"  Ref    log prob (rejected): {ref_l.tolist()}")
print(f"  Implicit reward margin β*(log_ratio_w - log_ratio_l):")
print(f"    {reward_margin.tolist()}")
print(f"  DPO Loss: {loss.item():.4f}")
print()
print("Effect of beta:")
for beta in [0.01, 0.05, 0.1, 0.2, 0.5, 1.0]:
    l, rm = dpo_loss(pi_w, pi_l, ref_w, ref_l, beta=beta)
    print(f"  β={beta:.2f}: loss={l.item():.4f}, mean_reward_margin={rm.mean().item():.4f}")

## 8. Group Relative Policy Optimization (GRPO)

GRPO (DeepSeek-R1, 2024) further simplifies RLHF by eliminating the value/critic network.

### Key Idea

For each prompt $x$, sample $G$ outputs $\{y_1, y_2, \ldots, y_G\}$ from the old policy and score them with a reward model. Use the **group mean** as the baseline:

$$\hat{A}_i = \frac{r_i - \mu_r}{\sigma_r}, \quad \mu_r = \frac{1}{G}\sum_{j=1}^G r_j, \quad \sigma_r = \text{std}(r_1, \ldots, r_G)$$

### GRPO Objective

Same PPO clip formula, but with group-relative advantages:

$$\mathcal{L}^{\text{GRPO}} = -\frac{1}{G}\sum_{i=1}^G \min\left(\frac{\pi_\theta(y_i|x)}{\pi_{\text{old}}(y_i|x)}\hat{A}_i,\ \text{clip}\left(\frac{\pi_\theta(y_i|x)}{\pi_{\text{old}}(y_i|x)}, 1-\varepsilon, 1+\varepsilon\right)\hat{A}_i\right) - \beta D_{KL}$$

### Why No Value Function?

The group mean $\mu_r$ acts as a Monte Carlo baseline — unbiased and requiring no learned critic. This saves memory (no separate value network) and simplifies training, at the cost of needing $G$ rollouts per prompt.

**Key property:** $\sum_i \hat{A}_i = 0$ when using mean normalization (advantages are zero-centered).

In [ ]:
def grpo_advantages(rewards):
    """Compute GRPO group-relative advantages.
    Args:
        rewards: (G,) or (B, G) rewards for G outputs per prompt
    Returns:
        advantages: same shape, normalized to zero mean unit std within group
    """
    mean = rewards.mean(dim=-1, keepdim=True)
    std  = rewards.std(dim=-1, keepdim=True) + 1e-8
    return (rewards - mean) / std


def grpo_loss(old_log_probs, new_log_probs, rewards, G=4, epsilon=0.2):
    """GRPO loss for a batch of prompts, each with G sampled outputs.
    Args:
        old_log_probs: (B, G) sequence log probs under old policy
        new_log_probs: (B, G) sequence log probs under current policy
        rewards:       (B, G) scalar rewards for each output
        G:             group size
        epsilon:       PPO clip threshold
    Returns:
        loss: scalar
        advantages: (B, G) group-relative advantages
    """
    # Group-relative advantage: normalize within each group
    advantages = grpo_advantages(rewards)   # (B, G)

    # Probability ratio
    ratio = torch.exp(new_log_probs - old_log_probs)   # (B, G)

    # PPO clipped objective
    obj_unclipped = ratio * advantages
    obj_clipped   = torch.clamp(ratio, 1 - epsilon, 1 + epsilon) * advantages
    clip_obj = torch.min(obj_unclipped, obj_clipped)

    loss = -clip_obj.mean()
    return loss, advantages


torch.manual_seed(42)
B, G = 3, 4   # 3 prompts, 4 outputs each

# Simulated rewards for G=4 outputs per prompt
rewards = torch.tensor([
    [0.2, 0.8, 0.5, 0.1],   # prompt 1: second output is best
    [0.9, 0.1, 0.3, 0.7],   # prompt 2: first output is best
    [0.4, 0.4, 0.4, 0.4],   # prompt 3: all equal (degenerate case)
])

advantages = grpo_advantages(rewards)

print("GRPO Group-Relative Advantages:")
print(f"Rewards:    {rewards.tolist()}")
print(f"Advantages: {advantages.round(decimals=3).tolist()}")
print()

# Verify advantages sum to (approximately) zero within each group
print("Sum of advantages per group (should be ~0):")
for i in range(B):
    print(f"  Prompt {i+1}: sum = {advantages[i].sum().item():.6f}")

# GRPO loss
old_log_probs = -torch.rand(B, G) * 5   # (B, G) sequence log probs
new_log_probs = old_log_probs + torch.randn(B, G) * 0.1  # small policy update

loss, adv = grpo_loss(old_log_probs, new_log_probs, rewards, G=G)
print(f"\nGRPO loss: {loss.item():.4f}")
print(f"Final advantages: {adv.round(decimals=3).tolist()}")

## 9. RLHF vs DPO vs GRPO Comparison

| Aspect | RLHF (PPO) | DPO | GRPO |
|---|---|---|---|
| **Needs reward model?** | Yes (separate RM) | No | Yes (or rule-based) |
| **Needs value function?** | Yes (critic) | No | No |
| **Training loop** | RL (complex) | Supervised-style | RL (simpler than PPO) |
| **Memory overhead** | High (policy + ref + RM + critic) | Low (policy + ref) | Medium (policy + ref + RM) |
| **Stability** | Tricky (reward hacking, instability) | Stable | More stable than PPO |
| **Data format** | Prompts (online rollouts) | Preference pairs | Prompts (G rollouts) |
| **KL constraint** | Per-token in reward | Implicit (log-ratio) | Explicit KL term |
| **Best for** | Complex reward shaping | Clean preference data | Reasoning tasks (R1) |

In [ ]:
def count_params(model):
    return sum(p.numel() for p in model.parameters())


# Simulate model sizes (mini versions for illustration)
d_model = 64
vocab_size = 100
T = 16

# Policy / SFT model (shared base for all approaches)
policy_model = nn.Sequential(
    nn.Embedding(vocab_size, d_model),
    nn.Linear(d_model, d_model),
    nn.ReLU(),
    nn.Linear(d_model, vocab_size)
)
policy_params = count_params(policy_model)

# Reference model (frozen copy of policy)
ref_params = policy_params  # same architecture

# Reward model (same arch as policy but scalar output)
reward_model = nn.Sequential(
    nn.Embedding(vocab_size, d_model),
    nn.Linear(d_model, d_model),
    nn.ReLU(),
    nn.Linear(d_model, 1)  # scalar output
)
rm_params = count_params(reward_model)

# Value/critic model (same arch as reward model)
value_model = nn.Sequential(
    nn.Embedding(vocab_size, d_model),
    nn.Linear(d_model, d_model),
    nn.ReLU(),
    nn.Linear(d_model, 1)
)
critic_params = count_params(value_model)

print("Component parameter counts (mini models for illustration):")
print(f"  Policy model:    {policy_params:>10,}")
print(f"  Reference model: {ref_params:>10,}  (frozen copy)")
print(f"  Reward model:    {rm_params:>10,}")
print(f"  Critic model:    {critic_params:>10,}")
print()

# Total params in memory for each approach
rlhf_ppo_total = policy_params + ref_params + rm_params + critic_params
dpo_total      = policy_params + ref_params
grpo_total     = policy_params + ref_params + rm_params

print("Total parameters loaded during training:")
print(f"  RLHF (PPO): policy + ref + RM + critic = {rlhf_ppo_total:>10,}  (baseline 1.0x)")
print(f"  DPO:        policy + ref               = {dpo_total:>10,}  ({dpo_total/rlhf_ppo_total:.2f}x)")
print(f"  GRPO:       policy + ref + RM          = {grpo_total:>10,}  ({grpo_total/rlhf_ppo_total:.2f}x)")
print()
print("Key takeaway: DPO requires the fewest loaded models — just policy + reference.")
print("GRPO drops the critic but keeps the RM; PPO requires all four components.")

## Summary

This chapter derived the mathematics of reinforcement learning as applied to LLMs:

| Concept | Mathematical Core | Practical Role |
|---|---|---|
| MDP for LLMs | $G_t = \sum_k \gamma^k R_{t+k}$ | Framework for RL |
| REINFORCE | $\nabla_\theta J = \mathbb{E}[\nabla \log \pi \cdot A_t]$ | Basic policy gradient |
| PPO | $\min(r_t A_t, \text{clip}(r_t) A_t)$ | Stable policy update |
| GAE | $\hat{A}_t = \sum_l (\gamma\lambda)^l \delta_{t+l}$ | Variance reduction |
| Reward Model | $-\log \sigma(r_w - r_l)$ | Learn human preferences |
| KL Penalty | $-\beta \log(\pi_\theta / \pi_{\text{ref}})$ | Prevent reward hacking |
| DPO | $-\log \sigma(\beta(\log r_w - \log r_l))$ | RL-free preference learning |
| GRPO | Group-normalized $r_i$ as baseline | No critic needed |

**Key insight:** Each method is a different tradeoff between computational cost, stability, and data requirements. DPO is simplest; PPO is most flexible; GRPO is the sweet spot for reasoning models.

**Next:** Chapter 10 covers Advanced Architectures & Efficiency — MoE, GQA, quantization, speculative decoding, and more.